In [1]:
using LinearAlgebra
using SparseArrays

In [2]:
NY=99
N=NY+1;  h=2.0/NY;
x=range(-1,stop=1,length=N)
e = ones(N);
a1=-2/2520/h;  a2=25/2520/h;  a3=-150/2520/h;  a4=600/2520/h;  a5=-2100/2520/h;
b5=2/2520/h;  b4=-25/2520/h;  b3=150/2520/h;  b2=-600/2520/h;  b1=2100/2520/h;
# L1 = spdiagm(N,N,[a1*e a2*e a3*e a4*e a5*e b1*e b2*e b3*e b4*e b5*e ], [-5,-4,-3,-2,-1,1,2,3,4,5]);
c=zeros(N)
L1 = diagm(-5=>a1*e,-4=>a2*e,-3=>a3*e,-2=>a4*e,-1=>a5*e,1=>b1*e,2=>b2*e,3=>b3*e,4=>b4*e,5=>b5*e,0=>c);
L1 = L1[1:N,1:N]
# L1=Matrix(L1)
  L1[1:5,:].=0; L1[N-4:N,:].=0;
  L1[1,1]=1
  L1[1,1] = -(25/12)/h;
  L1[1,2] = 48/12/h;
  L1[1,3] = -36/12/h;
  L1[1,4] = 16/12/h;
  L1[1,5] = -3/12/h;
  
  L1[2,1] = -3/12/h;
  L1[2,2] = -10/12/h;
  L1[2,3] = 18/12/h;
  L1[2,4] = -6/12/h;
  L1[2,5] = 1/12/h;
  
  L1[3,1] = 1/12/h;
  L1[3,2] = -8/12/h;
  L1[3,4] = 8/12/h;
  L1[3,5] = -1/12/h;

  L1[4,1] = -1/60/h;
  L1[4,2] = 9/60/h;
  L1[4,3] = -45/60/h;
  L1[4,5] = 45/60/h;
  L1[4,6] = -9/60/h;
  L1[4,7] = 1/60/h;
  
  c1 = -3.571428571428e-3/h;
  c2 = 3.80952380952e-2/h;
  c3 = -0.2/h;
  c4 = 0.8/h;
  
  L1[5,1] = -c1;
  L1[5,2] = -c2;
  L1[5,3] = -c3;
  L1[5,4] = -c4;
  L1[5,6] = c4;
  L1[5,7] = c3;
  L1[5,8] = c2;
  L1[5,9] = c1;
  
  L1[N,N] = 25/12/h;
  L1[N,N-1] = -48/12/h;
  L1[N,N-2] = 36/12/h;
  L1[N,N-3] = -16/12/h;
  L1[N,N-4] = 3/12/h;
  
  L1[N-1,N] = 3/12/h;
  L1[N-1,N-1] = 10/12/h;
  L1[N-1,N-2] = -18/12/h;
  L1[N-1,N-3] = 6/12/h;
  L1[N-1,N-4] = -1/12/h;
  
  L1[N-2,N] = -1/12/h;
  L1[N-2,N-1] = 8/12/h;
  L1[N-2,N-3] = -8/12/h;
  L1[N-2,N-4] = 1/12/h;
  
  L1[N-3,N] = 1/60/h;
  L1[N-3,N-1] = -9/60/h;
  L1[N-3,N-2] = 45/60/h;
  L1[N-3,N-4] = -45/60/h;
  L1[N-3,N-5] = 9/60/h;
  L1[N-3,N-6] = -1/60/h;
  
  L1[N-4,N] = c1;
  L1[N-4,N-1] = c2;
  L1[N-4,N-2] = c3;
  L1[N-4,N-3] = c4;
  L1[N-4,N-5] = -c4;
  L1[N-4,N-6] = -c3;
  L1[N-4,N-7] = -c2;
  L1[N-4,N-8] = -c1;
  L2=L1*L1;  sy=L1*x;  sy=1 ./sy;  syy=L1*sy;
  for k=1:NY+1
    L2[k,:] .= L2[k,:] .* [sy[k] .* sy[k]] .+ L1[k,:] .* [sy[k] .* syy[k]]
    L1[k,:] .= L1[k,:] .* [sy[k]]
  end

In [13]:
using BSplineKit
x=collect(x)
baseflow="Vonkarmen.txt"
N=99
ω=0
β=0.07759
R=285.36
Ro=1
Co=2
f = open( baseflow, "r" )
n = countlines( f )
seekstart( f )
data = zeros(n,6)
for i = 1:n
    z,w,u,v,du,dv = split( readline( f ), " " ) 
    data[i,1] = parse(Float64,z)
    data[i,2] = parse(Float64,w)
    data[i,3] = parse(Float64,u)
    data[i,4] = parse(Float64,v)  
    data[i,5] = parse(Float64,du)
    data[i,6] = parse(Float64,dv)
end

close( f )

t=data[:,1]
w=data[:,2]
u=data[:,3]
v=data[:,4]
du=data[:,5]
dv=data[:,6]
itpw=itp = interpolate(t, w , BSplineOrder(4))
itpu=itp = interpolate(t, u , BSplineOrder(4))
itpv=itp = interpolate(t, v , BSplineOrder(4))
itpdu=itp = interpolate(t, du , BSplineOrder(4))
itpdv=itp = interpolate(t, dv , BSplineOrder(4))
#interpolation
for i=1:N+1
    x[i,1]=10* x[i,1]+ 10
end
L1=0.1* L1
L2=0.01*L2

# for i=1:N+1
#     L1[i,:]=L1[i,:].*((2*x[i]^3-x[i]^2+3*x[i]-4)^2/(20*(6*x[i]^2-2*x[i]+3)))
#     L2[i,:]=L2[i,:].*((2*x[i]^3-x[i]^2+3*x[i]-4)^2/(20*(6*x[i]^2-2*x[i]+3)))^2
# end
# for i=1:N+1
#     x[i]=(4*x[i]^3-2*x[i]^2+6*x[i]+12)/(-2*x[i]^3+x[i]^2-3*x[i]+4)
#     if x[i]>20
#         x[i]=20
#     end
# end
D=L1
D2=L2
U=zeros(N+1,1)
V=zeros(N+1,1)
W=zeros(N+1,1)
dU=zeros(N+1,1)
dV=zeros(N+1,1)
dW=zeros(N+1,1)
p=zeros(N+1,1)
for i=1:N+1
    U[i,1]=itpu(x[i])
    V[i,1]=itpv(x[i])
    W[i,1]=itpw(x[i])
    dU[i,1]=itpdu(x[i])
    dV[i,1]=itpdv(x[i])
end
dW=-2*U
ddU=D*dU;
ddV=D*dV;
A=D2-(β^2)*I(N+1)
L0_1= im*A^2 + R*β*V.*A - R*ω.*A - R*β*ddV.*I(N+1) + im*ddU.*I(N+1) - Ro*im*W.*A*D - Ro*im*dW.*A - Ro*im*U.*D2
L0_2=(2Ro*V.+Co).*D + 2*Ro*dV.*I(N+1)
L0_3=(2Ro*V.+Co).*D+ im*R*β*dU.*I(N+1)
L0_4= im*A + R*β*V.*I(N+1) - R*ω*I(N+1) - im*Ro*W.*D - im*Ro*U.*I(N+1)
L1_1= -(1/R).*A + R*U.*A + im*β*V.*I(N+1) - im*ω*I(N+1) - R*ddU.*I(N+1) + Ro*(1/R)*W.*D + Ro*(1/R)*dW.*I(N+1)
L1_2= zeros(N+1,N+1)
L1_3= -1*im*R*dV.*I(N+1)
L1_4= R*U.*I(N+1)
L2_1= -2*im*A + ω*R*I(N+1) - β*R*V.*I(N+1) + im*U.*I(N+1) + Ro*im*W.*D + Ro*im*dW.*I(N+1)
L2_2= L1_2
L2_3= L1_2
L2_4= -im*I(N+1)
L3_1= (1/R)*I(N+1) - R*U.*I(N+1)
L3_2= L3_3=L3_4=L1_2
L4_1= im*I(N+1)
L4_2= L4_3=L4_4=L1_2
L2_4= Matrix{ComplexF64}(L2_4)
A0=[L0_1 L0_2 ;L0_3 L0_4 ;]
A1=[L1_1 L1_2 ;L1_3 L1_4 ;]
A2=[L2_1 L2_2 ;L2_3 L2_4 ;]
A3=[L3_1 L3_2 ;L3_3 L3_4 ;]
A4=[L4_1 L4_2 ;L4_3 L4_4 ;]
A0=A0[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A1=A1[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A2=A2[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A3=A3[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A4=A4[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A0=sparse(A0);
A1=sparse(A1);
A2=sparse(A2);
A3=sparse(A3);
A4=sparse(A4);

In [14]:
using NonlinearEigenproblems
nep = PEP([A0,A1,A2,A3,A4]); #Create a PEP object
sc=10
nep1 = shift_and_scale(nep,scale=sc);
mult_scale = norm(nep1.A[end]);
nep2 = PEP(nep1.A ./ mult_scale);
λ1,v1 = iar(nep2,σ=0.05,neigs=20,maxit=500);
λ_orig = sc*λ1

20-element Vector{ComplexF64}:
     0.1811066583449477 - 0.2798980543309142im
    0.22660719934603707 - 0.38963705026659823im
    0.35953694709329903 + 0.3679481801001924im
     0.3591918637268568 - 0.36668739950919876im
 5.0735735057649833e-11 - 0.10694819133649122im
  -6.637968022471519e-9 + 0.1069348063967518im
    0.18131736363488937 + 0.28128601174807455im
     0.2263386249926473 + 0.3910027324221488im
     0.4583456849570249 - 0.570440887375135im
      0.454527029399021 - 0.46238070478554943im
    0.45832615470343313 + 0.5718512140234882im
    0.36370765349101686 - 0.5540094390020084im
    0.45466402550154383 + 0.4637129659074928im
    0.36364094597767693 + 0.5553785600670744im
 1.4971475448266602e-10 - 0.3093398134573639im
    4.32727603749683e-9 + 0.30929656082642254im
     0.4229148189733786 - 0.7055769826436787im
     0.4228554239243621 + 0.7068972984954612im
    -1.6275358345752e-7 - 0.4558976811251426im
   9.057228154973895e-8 + 0.4564918429134611im